In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/global_suicide_rates_real_who_worldbank.csv")
df.head()

,country,country_code,year,sex,age_bracket,suicide_rate_per_100k,generation,gdp_usd,gdp_per_capita_usd,total_country_population
0,Afghanistan,AFG,2000,both,all_ages,4.36,All Cohorts,3.521418e+09,174.930991,20130327.0
1,Afghanistan,AFG,2000,female,all_ages,2.91,All Cohorts,3.521418e+09,174.930991,20130327.0
2,Afghanistan,AFG,2000,male,all_ages,5.79,All Cohorts,3.521418e+09,174.930991,20130327.0
3,Afghanistan,AFG,2001,both,all_ages,4.38,All Cohorts,2.813572e+09,138.706822,20284307.0
4,Afghanistan,AFG,2001,female,all_ages,2.92,All Cohorts,2.813572e+09,138.706822,20284307.0


In [3]:
# jumlah dan persentase kosong per kolom
na = df.isna().sum()
print(pd.DataFrame({"n": na, "pct": (na / len(df) * 100).round(1)}).query("n > 0"))

# kosongnya di negara mana saja
df[df.isna().any(axis=1)].groupby("country").size().sort_values(ascending=False).head(15)

                      n  pct
gdp_usd             315  1.7
gdp_per_capita_usd  315  1.7


country
Korea, Democratic People's Republic of    99
South Sudan                               75
Eritrea                                   63
Yemen                                     42
Cuba                                      36
dtype: int64

In [4]:
X = df.drop(columns=[
    "suicide_rate_per_100k"
])
y = df["suicide_rate_per_100k"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((14652, 9), (3663, 9), (14652,), (3663,))

In [5]:
preprocessor = ColumnTransformer([
    ('numeric', num_pipe(), ["year", "gdp_usd", "gdp_per_capita_usd", "total_country_population"]),
    ('categoric', cat_pipe(encoder='onehot'), ["country","country_code", "sex", "age_bracket","generation"]),
])

In [6]:
from xgboost import XGBRegressor
pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', XGBRegressor(n_jobs=-1, random_state=42, eval_metric='mlogloss'))
])

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(pipeline, rsp.xgb_params, cv=3, scoring='r2', n_iter=50, n_jobs=-1, verbose=1, random_state=42)
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
{'algo__colsample_bytree': np.float64(0.6439756413500356), 'algo__gamma': 4, 'algo__learning_rate': np.float64(0.7686119654652775), 'algo__max_depth': 7, 'algo__n_estimators': 158, 'algo__reg_alpha': np.float64(0.601367193359287), 'algo__reg_lambda': np.float64(3.325610295975196), 'algo__subsample': np.float64(0.6121770240668967)}
0.9731734148124814 0.8264214689395653 0.8333396431417662
